# Spread EMBIG Colombia — AR(1) sobre ln(spread), serie mensual

Flujo Box-Jenkins: **datos → transformación → identificación → estacionalidad → estimación → diagnóstico → pronóstico → ajuste por volatilidad → póster**.

- La serie es **mensual** (por defecto, **último dato del mes**) y se trabaja siempre en **ln(spread)**, sin diferenciar (d = 0).
- El único modelo es un **AR(1) con constante**.
- Cada etapa tiene sus funciones (arriba) y una llamada (abajo) que guarda lo que necesita el póster en el diccionario `R`.

## 0. Configuración

In [ ]:
from pathlib import Path
import warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

warnings.filterwarnings("ignore")

# --- modelo ---
ORDEN = (1, 0, 0)          # AR(1) sobre ln(spread)
TREND = "c"                # con constante: en statsmodels 'const' es la MEDIA de ln(spread)

# --- datos ---
HOJA = "Diarias"           # hoja del Excel (si tu base ya es mensual, pon aquí su hoja)
AGREGACION = "ultimo"      # "ultimo" = último dato del mes | "promedio" = promedio mensual
EXCLUIR_MES_INCOMPLETO = True   # descarta el último mes si los datos terminan antes de su cierre
S = 12                     # periodos por año (estacionalidad mensual)

# --- pronóstico y salidas ---
H = 6                      # horizonte en meses (6 meses = 2 trimestres)
PUNTO = "mediana"          # "mediana" = exp(μ) (consistente con los IC) | "media" = exp(μ + σ²/2)
VENTANA_GRAFICO = 60       # meses de historia que se muestran en los gráficos de pronóstico
LAGS_LB = [6, 12, 18, 24]  # rezagos del Ljung-Box
LAGS_ARCH = [1, 3, 6]      # rezagos del ARCH-LM
AUTORES = ""               # <-- editar

# --- rutas ---
BASE_DIR = next((r for r in (Path.cwd(), *Path.cwd().parents)
                 if (r / "base de datos" / "Datos_EMBI.xlsx").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("No se encontró base de datos/Datos_EMBI.xlsx en la carpeta del proyecto ni en sus carpetas superiores.")
ARCHIVO = BASE_DIR / "base de datos" / "Datos_EMBI.xlsx"
SALIDA = BASE_DIR / "code" / "resultados_embi"
SALIDA.mkdir(parents=True, exist_ok=True)

# --- estilo gráfico claro (figuras y póster) ---
plt.style.use("default")
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
                     "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
C_AZUL, C_VERDE, C_ROJO = "#1F4E78", "#2E7D32", "#C0392B"

R = {}                     # resultados de cada etapa; el póster solo lee este diccionario

## 1. Datos: serie mensual

In [ ]:
def cargar_serie(archivo, hoja):
    """Lee fecha y valor (dos primeras columnas) y devuelve la serie indexada por fecha; omite filas inválidas."""
    raw = pd.read_excel(archivo, sheet_name=hoja, header=1)
    col_fecha, col_valor = raw.columns[:2]
    fecha = pd.to_datetime(raw[col_fecha], errors="coerce")
    valor = pd.to_numeric(raw[col_valor], errors="coerce")
    validas = fecha.notna() & valor.notna()
    if not validas.any():
        raise ValueError("No se encontraron filas con fecha y spread válidos.")
    if not validas.all():
        print(f"Filas omitidas por fecha o spread inválido/vacío: {(~validas).sum()}")
    return pd.Series(valor[validas].to_numpy(), index=pd.DatetimeIndex(fecha[validas]), name="spread").sort_index()

def a_mensual(serie, agregacion="ultimo", excluir_incompleto=True):
    """Agrega a mensual (índice = primer día del mes). Funciona igual si la serie ya es mensual."""
    if excluir_incompleto:
        ult = serie.index.max()
        if (ult + pd.offsets.MonthEnd(0) - ult).days > 5:         # los datos terminan lejos del cierre del mes
            print(f"Se descarta {ult:%m/%Y}: el último dato es del {ult:%d/%m/%Y} (mes incompleto).")
            serie = serie[serie.index < ult.replace(day=1)]
    g = serie.resample("MS")
    m = (g.last() if agregacion == "ultimo" else g.mean())
    faltan = int(m.isna().sum())
    if faltan:
        print(f"Aviso: {faltan} mes(es) sin datos; se omiten.")
    return m.dropna().rename("spread")

diaria = cargar_serie(ARCHIVO, HOJA)
y = a_mensual(diaria, AGREGACION, EXCLUIR_MES_INCOMPLETO)     # spread mensual (pbs)
ly = np.log(y)                                                # ln(spread): serie del modelo (d = 0)

etq_agr = "último dato del mes" if AGREGACION == "ultimo" else "promedio mensual"
R.update(n=len(y), ini=f"{y.index[0]:%m/%Y}", fin=f"{y.index[-1]:%m/%Y}", ultimo=float(y.iloc[-1]), agr=etq_agr)
print(f"Datos originales: {diaria.index.min():%d/%m/%Y} a {diaria.index.max():%d/%m/%Y} ({len(diaria)} filas)")
print(f"Serie mensual ({etq_agr}): {len(y)} obs., {y.index[0]:%m/%Y} a {y.index[-1]:%m/%Y}")

## 2. Transformación y varianza

In [ ]:
def grafico_transformacion(y, ly, ruta):
    fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    ax[0].plot(y.index, y, color=C_AZUL, lw=1.6)
    ax[0].set_title("ANTES: Spread EMBIG Colombia (pbs), mensual"); ax[0].set_ylabel("pbs")
    ax[1].plot(ly.index, ly, color=C_VERDE, lw=1.6); ax[1].axhline(ly.mean(), color="gray", lw=.8, ls="--")
    ax[1].set_title("DESPUÉS: ln(spread) (serie del AR(1); línea = media)"); ax[1].set_ylabel("ln")
    plt.tight_layout(); fig.savefig(ruta, dpi=150); plt.show()

def tabla_varianza(series):
    """Levene entre mitades y ARCH(1), ARCH(4) sobre residuos de un AR(1) auxiliar."""
    def arch_ar1(s, lags):
        s = np.asarray(s)
        return het_arch(sm.OLS(s[1:], sm.add_constant(s[:-1])).fit().resid, nlags=lags)[1]
    for nombre, s in series.items():
        s = s.reset_index(drop=True); h = len(s) // 2
        print(f"{nombre:6s} var1={s[:h].var():.4f} var2={s[h:].var():.4f} "
              f"Levene p={stats.levene(s[:h], s[h:]).pvalue:.4f} | "
              f"ARCH(1) p={arch_ar1(s,1):.4f} ARCH(4) p={arch_ar1(s,4):.4f}")

grafico_transformacion(y, ly, SALIDA / "fig1_transformacion.png")
print("--- Varianza constante ---")
tabla_varianza({"Nivel": y, "ln": ly})

## 3. Etapa 1 — Identificación sobre ln(spread)

In [ ]:
def prueba_adf(s, reg):
    r = adfuller(s, regression=reg, autolag="AIC")
    print(f"ADF  reg={reg:2s} est={r[0]:.3f} p={r[1]:.4f} rezagos={r[2]} crit5%={r[4]['5%']:.3f}")
    return r[1]

def prueba_kpss(s, reg):
    r = kpss(s, regression=reg, nlags=int(4 * (len(s) / 100) ** 0.25))
    print(f"KPSS reg={reg:2s} est={r[0]:.3f} p={r[1]:.4f} crit5%={r[3]['5%']}")
    return r[1]

def grafico_fac_facp(y, ly, ruta, lags=36):
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    for fila, (s, nom) in enumerate([(y, "spread original"), (ly, "ln(spread)")]):
        plot_acf(s, lags=lags, ax=ax[fila, 0], title=f"FAC – {nom}")
        plot_pacf(s, lags=lags, ax=ax[fila, 1], method="ywm", title=f"FACP – {nom}")
    plt.tight_layout(); fig.savefig(ruta, dpi=150); plt.show()

print("=== ETAPA 1: IDENTIFICACIÓN (ln(spread)) ===")
adf_ly = prueba_adf(ly, "c"); kpss_ly = prueba_kpss(ly, "c")
adf_ct = prueba_adf(ly, "ct"); kpss_ct = prueba_kpss(ly, "ct")
grafico_fac_facp(y, ly, SALIDA / "fig2_FAC_FACP.png")
R["ident"] = dict(adf_ly=adf_ly, kpss_ly=kpss_ly, adf_ct=adf_ct, kpss_ct=kpss_ct)

## 4. Etapa 1b — Estacionalidad mensual (s = 12) sobre el AR(1)

In [ ]:
def test_estacionalidad(ly, s=S, n_perm=300, semilla=0):
    """LR de dummies mensuales en el AR(1), Kruskal-Wallis por mes, FAC en rezagos s, 2s, 3s y fuerza STL."""
    per = ly.index.month if s == 12 else ly.index.quarter
    base = ARIMA(ly, order=(1, 0, 0), trend="c").fit()
    X = pd.get_dummies(per, drop_first=True).astype(float); X.index = ly.index
    est = ARIMA(ly, exog=X, order=(1, 0, 0), trend="c").fit()
    lr = 2 * (est.llf - base.llf)
    assert lr >= 0, "LR negativo: el modelo con dummies no convergió"
    gl = X.shape[1]
    res = pd.Series(base.resid, index=ly.index)
    p_kw = stats.kruskal(*[res[res.index.month == k] if s == 12 else res[res.index.quarter == k]
                           for k in range(1, s + 1)]).pvalue
    a = acf(res, nlags=3 * s)

    def fuerza(r):
        st = STL(r, period=s, robust=True).fit()
        return max(0, 1 - st.resid.var() / (st.seasonal + st.resid).var())
    fs = fuerza(res)
    rng = np.random.default_rng(semilla)               # calibración: residuos barajados no tienen estacionalidad
    sims = np.array([fuerza(pd.Series(rng.permutation(res.values), index=res.index)) for _ in range(n_perm)])
    return dict(lr=lr, gl=gl, p_lr=stats.chi2.sf(lr, df=gl), p_kw=p_kw, acf=a[[s, 2 * s, 3 * s]],
                banda=1.96 / np.sqrt(len(res)), fs=fs, p_perm=float(np.mean(sims >= fs)), fs_tipica=float(np.median(sims)))

print(f"=== ETAPA 1b: ESTACIONALIDAD EN AR(1), s = {S} ===")
E = test_estacionalidad(ly)
print(f"LR dummies: LR={E['lr']:.2f}, gl={E['gl']}, p={E['p_lr']:.4f}")
print(f"Kruskal-Wallis por periodo: p={E['p_kw']:.4f}")
print(f"FAC residuos, rezagos {S}, {2*S}, {3*S}:", E["acf"].round(3), "| banda ±", round(E["banda"], 3))
print(f"Fuerza STL={E['fs']:.3f} | p permutación={E['p_perm']:.3f} | típica sin estacionalidad={E['fs_tipica']:.3f}")
E["hay"] = min(E["p_lr"], E["p_kw"]) < 0.05
R["estac"] = E

## 5. Etapa 2 — Estimación del AR(1)

In [ ]:
def ajustar_ar1(ly):
    """AR(1) con constante por máxima verosimilitud. En statsmodels, 'const' es la media de ln(spread)."""
    m = ARIMA(ly, order=ORDEN, trend=TREND).fit()
    phi, mu = m.params["ar.L1"], m.params["const"]
    ic = m.conf_int().loc["ar.L1"].to_numpy()
    info = dict(phi=phi, se_phi=m.bse["ar.L1"], p_phi=m.pvalues["ar.L1"], ic_phi=ic, ic_incluye_1=bool(ic[1] >= 1),
                mu=mu, se_mu=m.bse["const"], p_mu=m.pvalues["const"], sigma2=m.params["sigma2"],
                vida_media=np.log(.5) / np.log(phi), nivel_lp=float(np.exp(mu)), aic=m.aic, bic=m.bic)
    return m, info

print("=== ETAPA 2: ESTIMACIÓN ===")
modelo, M = ajustar_ar1(ly)
print(modelo.summary().tables[1])
print(f"φ={M['phi']:.3f} (EE {M['se_phi']:.3f})  IC95%=[{M['ic_phi'][0]:.3f}, {M['ic_phi'][1]:.3f}]  (¿incluye 1? {'sí' if M['ic_incluye_1'] else 'no'})")
print(f"μ (media de ln)={M['mu']:.3f} (EE {M['se_mu']:.3f}) | σ²={M['sigma2']:.4f}")
print(f"Vida media={M['vida_media']:.1f} meses | nivel de largo plazo={M['nivel_lp']:.0f} pbs | AIC={M['aic']:.2f} BIC={M['bic']:.2f}")
R["modelo"] = M

## 6. Etapa 3 — Diagnóstico de residuos

In [ ]:
def diagnosticar(m):
    """Pruebas sobre los residuos: Ljung-Box, ARCH-LM, Jarque-Bera."""
    res = np.asarray(m.resid)
    p, _, q = m.model.order
    lb = acorr_ljungbox(res, lags=LAGS_LB, model_df=p + q)["lb_pvalue"].to_numpy()
    arch = np.array([het_arch(res, nlags=k)[1] for k in LAGS_ARCH])
    return dict(lb_p=lb, arch_p=arch, jb_p=stats.jarque_bera(res)[1],
                asim=stats.skew(res), curt=stats.kurtosis(res, fisher=False))

def grafico_residuos(m, ruta, lags=36):
    """Residuales, FAC, FAC de residuos² (volatilidad) y Q-Q (normalidad)."""
    res = pd.Series(np.asarray(m.resid), index=ly.index)
    fig, ax = plt.subplots(2, 2, figsize=(11, 7))
    ax[0, 0].plot(res, color="#444", lw=1); ax[0, 0].axhline(0, color="gray", lw=.8); ax[0, 0].set_title("Residuales AR(1)")
    plot_acf(res, lags=lags, ax=ax[0, 1], title="FAC residuos")
    plot_acf(res ** 2, lags=lags, ax=ax[1, 0], title="FAC residuos² (volatilidad)")
    stats.probplot(res, dist="norm", plot=ax[1, 1]); ax[1, 1].set_title("Q-Q residuos (normalidad)")
    plt.tight_layout(); fig.savefig(ruta, dpi=150); plt.show()

print("=== ETAPA 3: DIAGNÓSTICO ===")
D = diagnosticar(modelo)
print(f"Ljung-Box p {LAGS_LB}:", D["lb_p"].round(3).tolist())
print(f"ARCH-LM p {LAGS_ARCH}:", D["arch_p"].round(3).tolist())
print(f"Jarque-Bera p: {D['jb_p']:.2e} | asimetría={D['asim']:.2f} curtosis={D['curt']:.2f}")
grafico_residuos(modelo, SALIDA / "fig3_residuos.png")
R["diag"] = D

## 7. Etapa 4 — Pronóstico (IC 95% y 90%, en pbs)

In [ ]:
def pronosticar(modelo, fechas, h=H, punto="mediana"):
    """Mediana = exp(μ_h); media = exp(μ_h + σ²_h/2) bajo normalidad en logs. Los IC corresponden a la mediana."""
    fc = modelo.get_forecast(h)
    mu_h, var_h = fc.predicted_mean.values, np.asarray(fc.var_pred_mean)
    mediana, media = np.exp(mu_h), np.exp(mu_h + 0.5 * var_h)
    fut = [fechas[-1] + pd.DateOffset(months=i + 1) for i in range(h)]
    return dict(fc=fc, fut=fut, mediana=mediana, media=media, etq=punto,
                punto=mediana if punto == "mediana" else media,
                ic95=np.exp(fc.conf_int(alpha=0.05).values), ic90=np.exp(fc.conf_int(alpha=0.10).values))

def panel_pronostico(ax, y, P, a95, a90, color, titulo, ref95=None, nivel_lp=None):
    fut, punto = P["fut"], P["punto"]
    ax.plot(y.index[-VENTANA_GRAFICO:], y[-VENTANA_GRAFICO:], color=C_AZUL, lw=1.4, label="Observado")
    ax.plot([y.index[-1], *fut], [y.iloc[-1], *punto], "--o", ms=3, color=color, label=f"Pronóstico ({P['etq']})")
    ax.fill_between(fut, a95[:, 0], a95[:, 1], color=color, alpha=.15, label="IC 95%")
    ax.fill_between(fut, a90[:, 0], a90[:, 1], color=color, alpha=.30, label="IC 90%")
    if ref95 is not None:
        ax.plot(fut, ref95[:, 0], ":", color="gray", lw=1.3, label="IC 95% original"); ax.plot(fut, ref95[:, 1], ":", color="gray", lw=1.3)
    if nivel_lp is not None:
        ax.axhline(nivel_lp, color="gray", ls=":", lw=1, label="Nivel de largo plazo")
    ax.annotate(f"{punto[-1]:.0f}", xy=(fut[-1], punto[-1]), xytext=(0, 5), textcoords="offset points",
                ha="center", color=color, fontsize=8, fontweight="bold", zorder=10)
    ax.set_title(titulo, fontsize=10); ax.set_ylabel("pbs"); ax.legend(fontsize=8)

print("=== ETAPA 4: PRONÓSTICO ===")
P = pronosticar(modelo, y.index, punto=PUNTO)
pron = pd.DataFrame({"Fecha": [f"{f:%m/%Y}" for f in P["fut"]], "Mediana": P["mediana"], "Media": P["media"],
                     "IC95_inf": P["ic95"][:, 0], "IC95_sup": P["ic95"][:, 1],
                     "IC90_inf": P["ic90"][:, 0], "IC90_sup": P["ic90"][:, 1]})
print(pron.round(1).to_string(index=False))
print(f"Último dato ({y.index[-1]:%m/%Y}): {y.iloc[-1]:.0f} pbs | nivel de largo plazo: {M['nivel_lp']:.0f} pbs")
print(f"Se reporta la {PUNTO}. Los IC son los de la mediana.")
fig, ax = plt.subplots(figsize=(10, 5))
panel_pronostico(ax, y, P, P["ic95"], P["ic90"], C_ROJO, "Pronóstico Spread EMBIG Colombia – AR(1) sobre ln", nivel_lp=M["nivel_lp"])
plt.tight_layout(); fig.savefig(SALIDA / "fig4_pronostico.png", dpi=150); plt.show()
R["pron"] = P

## 8. Etapa 5 — Intervalos ajustados por volatilidad

Solo cambia el **ancho** de los IC: el modelo, los coeficientes y el pronóstico puntual no se modifican. Se detecta el corte donde cambia la varianza de los residuos, se calcula σ de la época reciente y se reescala el error estándar del pronóstico por σ_reciente / σ_total. El rango ajustado **supone que no habrá una nueva crisis**: preséntalo junto al original.

In [ ]:
CORTE_VOL = None     # None = detectar automáticamente; o fijar, p. ej. pd.Timestamp(2009, 3, 1)
MIN_OBS = 60         # mínimo de residuos (meses) a cada lado del corte

def ajustar_volatilidad(modelo, P, fechas, corte=None, min_obs=MIN_OBS):
    res = pd.Series(np.asarray(modelo.resid), index=fechas)
    if corte is None:
        cand = [(res.index[k], stats.levene(res.iloc[:k], res.iloc[k:]).pvalue) for k in range(min_obs, len(res) - min_obs)]
        corte = min(cand, key=lambda t: t[1])[0]
    ant, rec = res[res.index < corte], res[res.index >= corte]
    sig_tot, sig_rec = np.sqrt(modelo.params["sigma2"]), rec.std()
    factor = sig_rec / sig_tot
    mu_f, se = np.asarray(P["fc"].predicted_mean), np.asarray(P["fc"].se_mean) * factor
    def ic(alpha):
        z = stats.norm.ppf(1 - alpha / 2)
        return np.exp(np.column_stack([mu_f - z * se, mu_f + z * se]))
    return dict(corte=corte, lev=stats.levene(ant, rec).pvalue, sig_ant=ant.std(), sig_rec=sig_rec,
                sig_tot=sig_tot, factor=factor, ic95=ic(0.05), ic90=ic(0.10),
                jb_rec=stats.jarque_bera(rec)[1], n_rec=len(rec))

print("=== ETAPA 5: AJUSTE DE VOLATILIDAD ===")
V = ajustar_volatilidad(modelo, P, y.index, CORTE_VOL)
print(f"Corte: {V['corte']:%m/%Y} | σ antes={V['sig_ant']:.3f} σ después={V['sig_rec']:.3f} | Levene p={V['lev']:.4f}")
print(f"Época reciente: n={V['n_rec']} | Jarque-Bera p={V['jb_rec']:.3f}")
print(f"σ total={V['sig_tot']:.3f} | σ reciente={V['sig_rec']:.3f} | factor={V['factor']:.2f}")
comp = pd.DataFrame({"Fecha": pron["Fecha"], P["etq"].capitalize(): P["punto"],
                     "IC95_orig": [f"{a:.0f}-{b:.0f}" for a, b in P["ic95"]], "IC95_ajust": [f"{a:.0f}-{b:.0f}" for a, b in V["ic95"]],
                     "IC90_orig": [f"{a:.0f}-{b:.0f}" for a, b in P["ic90"]], "IC90_ajust": [f"{a:.0f}-{b:.0f}" for a, b in V["ic90"]]})
print(comp.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
panel_pronostico(ax, y, P, V["ic95"], V["ic90"], C_VERDE, f"Pronóstico AR(1) – intervalos ajustados (σ desde {V['corte']:%m/%Y})", ref95=P["ic95"], nivel_lp=M["nivel_lp"])
plt.tight_layout(); fig.savefig(SALIDA / "fig5_pronostico_ajustado.png", dpi=150); plt.show()
fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
panel_pronostico(axs[0], y, P, P["ic95"], P["ic90"], C_ROJO, "ORIGINAL: σ de toda la muestra")
panel_pronostico(axs[1], y, P, V["ic95"], V["ic90"], C_VERDE, f"AJUSTADO: σ desde {V['corte']:%m/%Y}")
plt.tight_layout(); fig.savefig(SALIDA / "fig6_comparacion_original_vs_ajustado.png", dpi=150); plt.show()
R["vol"] = V

## 9. Etapa 6 — Póster (PDF, paleta clara)

In [ ]:
# Paleta clara
BG, CARD, BORDE, PILL = "#F4F7FB", "#FFFFFF", "#D5DCEB", "#CFE3FF"
INK, MUTED, ACC, OK, KO, HEAD = "#1B2A4A", "#5A6785", "#D97706", "#1E9E5A", "#D64545", "#DCE8FA"

def textos_poster(R):
    """Frases del póster, construidas con las cifras y p-valores de R (cambian solas si cambian los datos)."""
    I, E, M, D, V, P = R["ident"], R["estac"], R["modelo"], R["diag"], R["vol"], R["pron"]
    raiz = ("se rechaza raíz unitaria" if I["adf_ly"] < 0.05 else
            "no se rechaza raíz unitaria; el test tiene poca potencia con φ cercano a 1")
    ident = (f"Serie mensual ({R['agr']}). El spread en niveles tiene varianza cambiante, por lo que se usa ln. "
             f"ADF sobre ln(spread): p = {I['adf_ly']:.3f} ({raiz}); KPSS: p = {I['kpss_ly']:.3f}. "
             f"La FACP de ln(spread) tiene un pico dominante en el rezago 1 (patrón AR(1)), así que se modela en niveles de ln (d = 0). "
             f"φ = {M['phi']:.3f}, IC 95% [{M['ic_phi'][0]:.2f}, {M['ic_phi'][1]:.2f}] {'incluye' if M['ic_incluye_1'] else 'no incluye'} el 1.")
    est = (f"Estacionalidad mensual (s=12) sobre el AR(1): LR de dummies p = {E['p_lr']:.2f}, Kruskal-Wallis p = {E['p_kw']:.2f}. "
           + ("Hay evidencia de patrón estacional; se contrasta un SARIMA." if E["hay"] else
              "No hay evidencia de estacionalidad; no se justifica un SARIMA."))
    modelo = (f"Vida media de un choque: {M['vida_media']:.1f} meses. Nivel de largo plazo: {M['nivel_lp']:.0f} pbs "
              f"(promedio de toda la muestra, crisis incluidas).")
    sup = ("Los IC asumen errores normales; la normalidad se rechaza (colas pesadas por las crisis), por eso se complementan con un rango ajustado por volatilidad."
           if D["jb_p"] < 0.05 else "Los residuos son compatibles con errores normales.")
    pron = (f"Último dato: {R['ultimo']:.0f} pbs ({R['fin']}). Se reporta la {P['etq']}; los IC son los de la mediana. Rango ajustado: σ desde {V['corte']:%m/%Y} "
            f"(σ = {V['sig_rec']:.3f} vs {V['sig_tot']:.3f}, Levene p = {V['lev']:.3f}); supone que no habrá una nueva crisis. "
            f"El pronóstico revierte hacia el nivel de largo plazo.")
    return dict(ident=ident, est=est, modelo=modelo, sup=sup, pron=pron)

def armar_poster(R, carpeta, salida_pdf, salida_png=None):
    T = textos_poster(R)
    M, D, P, V = R["modelo"], R["diag"], R["pron"], R["vol"]
    fig = plt.figure(figsize=(16.5, 23.4), facecolor=BG)
    bg = fig.add_axes([0, 0, 1, 1]); bg.set_xlim(0, 1); bg.set_ylim(0, 1); bg.axis("off")

    def card(x0, y0, w, h, titulo):
        bg.add_patch(FancyBboxPatch((x0, y0), w, h, boxstyle="round,pad=0,rounding_size=0.012", fc=CARD, ec=BORDE, lw=1.5))
        bg.text(x0 + 0.02, y0 + h - 0.006, titulo, color=INK, fontsize=22, fontweight="bold", va="center",
                bbox=dict(boxstyle="round,pad=0.5", fc=PILL, ec="none"))
    def texto(x, yv, s, w=60, size=15, color=INK, **kw):
        bg.text(x, yv, textwrap.fill(s, w), color=color, fontsize=size, va="top", ha="left", **kw)
    def imagen(rect, nombre):
        ax = fig.add_axes(rect); ax.imshow(plt.imread(carpeta / nombre)); ax.axis("off")

    # encabezado
    bg.add_patch(Rectangle((0, 0.915), 1, 0.085, fc=HEAD, ec="none"))
    bg.text(0.5, 0.968, "SPREAD EMBIG DE COLOMBIA", color=INK, fontsize=44, fontweight="bold", ha="center", va="center")
    bg.text(0.5, 0.936, f"Modelo AR(1) sobre ln(spread) | serie mensual {R['ini']} – {R['fin']} ({R['n']} obs.)",
            color=ACC, fontsize=20, ha="center", va="center", fontweight="bold")
    bg.text(0.03, 0.985, AUTORES, color=MUTED, fontsize=13, ha="left", va="center")

    # 1. serie original e identificación
    card(0.02, 0.655, 0.96, 0.250, "Serie original e identificación")
    imagen([0.03, 0.695, 0.34, 0.175], "fig1_transformacion.png")
    imagen([0.39, 0.695, 0.58, 0.175], "fig2_FAC_FACP.png")
    texto(0.03, 0.688, T["ident"], w=150, size=13)

    # 2. estimación y estacionalidad
    card(0.02, 0.490, 0.96, 0.155, "Estimación y estacionalidad")
    ee = lambda v: f"({v:.3f})"
    tabla = (f"{'Parámetro':16s}{'Estim.':>9s}{'(EE)':>10s}{'p':>8s}\n"
             f"{'φ  (AR1)':16s}{M['phi']:9.3f}{ee(M['se_phi']):>10s}{M['p_phi']:8.3f}\n"
             f"{'μ  (media de ln)':16s}{M['mu']:9.3f}{ee(M['se_mu']):>10s}{M['p_mu']:8.3f}\n"
             f"{'σ²':16s}{M['sigma2']:9.4f}\n"
             f"{'AIC / BIC':16s}{M['aic']:9.1f}{M['bic']:10.1f}")
    bg.text(0.03, 0.598, tabla, color=INK, fontsize=14, family="monospace", va="top")
    texto(0.03, 0.525, T["modelo"], w=70, size=13, color=MUTED)
    texto(0.55, 0.598, T["est"], w=55, size=15)

    # 3. verificación de supuestos
    card(0.02, 0.270, 0.96, 0.210, "Verificación de supuestos")
    tests = [("Sin autocorrelación", f"Ljung-Box {LAGS_LB}: p = {', '.join(f'{v:.2f}' for v in D['lb_p'])}", D["lb_p"].min() > 0.05),
             ("Homocedasticidad", f"ARCH-LM {LAGS_ARCH}: p = {', '.join(f'{v:.2f}' for v in D['arch_p'])}", D["arch_p"].min() > 0.05),
             ("Normalidad", f"Jarque-Bera: p = {D['jb_p']:.1e}", D["jb_p"] > 0.05)]
    for i, (nom, det, ok) in enumerate(tests):
        yy = 0.430 - i * 0.038
        bg.text(0.03, yy, "✓" if ok else "✗", color=OK if ok else KO, fontsize=26, fontweight="bold", va="center")
        bg.text(0.055, yy + 0.008, nom, color=INK, fontsize=16, fontweight="bold", va="center")
        bg.text(0.055, yy - 0.010, det, color=MUTED, fontsize=13, va="center")
    texto(0.03, 0.322, T["sup"], w=62, size=13, color=MUTED)
    imagen([0.50, 0.280, 0.47, 0.185], "fig3_residuos.png")

    # 4. pronóstico (horizontes mostrados: primero, intermedio y último)
    card(0.02, 0.090, 0.96, 0.170, "Pronóstico (pbs)")
    et = P["etq"].capitalize()
    bg.text(0.03, 0.212, f"{'Mes':10s}{et:>9s}{'IC95 orig.':>16s}{'IC95 ajust.':>16s}", color=ACC, fontsize=14, family="monospace", va="top", fontweight="bold")
    for j, i in enumerate(sorted({0, len(P["fut"]) // 2 - 1, len(P["fut"]) - 1})):
        a, b, c, d = P["ic95"][i, 0], P["ic95"][i, 1], V["ic95"][i, 0], V["ic95"][i, 1]
        bg.text(0.03, 0.192 - j * 0.020, f"{P['fut'][i]:%m/%Y}  {P['punto'][i]:8.0f}  {f'[{a:.0f}, {b:.0f}]':>15s}  {f'[{c:.0f}, {d:.0f}]':>15s}",
                color=INK, fontsize=14, family="monospace", va="top")
    texto(0.03, 0.127, T["pron"], w=72, size=12, color=MUTED)
    imagen([0.50, 0.095, 0.47, 0.150], "fig5_pronostico_ajustado.png")

    # 5. resultados
    card(0.02, 0.012, 0.96, 0.068, "Resultados")
    bg.text(0.5, 0.030, rf"$\ln S_t-{M['mu']:.3f}={M['phi']:.3f}\,(\ln S_{{t-1}}-{M['mu']:.3f})+\varepsilon_t$",
            color=INK, fontsize=26, ha="center", va="center")
    bg.text(0.98, 0.003, "Datos: Banco de la República (PD04715XD)", color=MUTED, fontsize=10, ha="right")

    fig.savefig(salida_pdf, facecolor=BG)
    if salida_png: fig.savefig(salida_png, dpi=60, facecolor=BG)
    plt.show()

armar_poster(R, SALIDA, SALIDA / "poster_embig.pdf", SALIDA / "poster_embig_preview.png")